# 16. Signal Processing & Mathematical Transforms: Beginner Guide

### 🌟 What Are Mathematical Transforms & Polynomial Fitting?
NumPy includes powerful numerical algorithms for signal processing and curve fitting, including the Fast Fourier Transform (`np.fft.fft`) for frequency analysis and least-squares polynomial regression (`np.polyfit`) for trendline modeling.

This interactive guide loads and works directly with `data/raw_transactions.csv`, giving you real-world hands-on practice.

### 📚 Key Concepts Covered in this Notebook:
- **Fast Fourier Transforms**: Covers `np.fft.fft()`, `np.fft.ifft()`, and `np.fft.fftfreq()`.
- **Curve Fitting & Polynomials**: Covers `np.polyfit()` and `np.polyval()`.


In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import numpy as np
import pandas as pd
import sys
import time
import os

# Load raw transactions and extract aligned NumPy numeric arrays
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
raw_df = pd.read_csv(csv_path)
clean_raw = raw_df.dropna(subset=['transaction_amount', 'is_fraud', 'account_age_months']).reset_index(drop=True)
amounts = clean_raw['transaction_amount'].to_numpy(dtype=np.float64)
fraud_flags = clean_raw['is_fraud'].to_numpy(dtype=np.int8)
account_ages = clean_raw['account_age_months'].to_numpy(dtype=np.float32)

print(f"NumPy Version: {np.__version__}")
print(f"Loaded from {csv_path} ({len(amounts)} clean aligned rows):")
print(f"- amounts array: shape {amounts.shape}, dtype {amounts.dtype}")
print(f"- fraud_flags array: shape {fraud_flags.shape}, dtype {fraud_flags.dtype}")
print(f"- account_ages array: shape {account_ages.shape}, dtype {account_ages.dtype}")

NumPy Version: 1.26.4
Loaded from ../data/raw_transactions.csv (14262 clean aligned rows):
- amounts array: shape (14262,), dtype float64
- fraud_flags array: shape (14262,), dtype int8
- account_ages array: shape (14262,), dtype float32


### 🔹 1D Fast Fourier Transform with `np.fft.fft()`
Decomposes daily transaction volume into cyclical frequency components. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `np.fft.fft(daily_signal)`


In [2]:
daily_signal = amounts[:512]  # Power of 2 for optimal FFT
fft_coeffs = np.fft.fft(daily_signal)
print('FFT Complex Coefficients (first 3):', fft_coeffs[:3])

FFT Complex Coefficients (first 3): [530491.05         +0.j          -5274.99438097+1765.65829472j
   2838.34008275-2268.26429804j]


### 🔹 Sample Frequencies with `np.fft.fftfreq()`
Identifies dominant cyclical spending periodicities. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `np.fft.fftfreq(len(daily_signal))`


In [3]:
freqs = np.fft.fftfreq(len(daily_signal))
peak_idx = np.argmax(np.abs(fft_coeffs[1:len(daily_signal)//2])) + 1
print(f'Dominant Frequency Detected: {freqs[peak_idx]:.4f} cycles/transaction')

Dominant Frequency Detected: 0.3027 cycles/transaction


### 🔹 Inverse FFT Denoising with `np.fft.ifft()`
Denoises the transaction volume curve by zeroing high-frequency noise and reconstructing the signal. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `np.fft.ifft(filtered_fft).real`


In [4]:
filtered_fft = fft_coeffs.copy()
filtered_fft[np.abs(freqs) > 0.1] = 0.0  # Zero high frequencies
denoised_ts = np.fft.ifft(filtered_fft).real
print('Denoised Transaction Signal Head:', denoised_ts[:4].round(2))

Denoised Transaction Signal Head: [702.6  650.78 624.32 619.84]


### 🔹 Least-Squares Polynomial Fitting with `np.polyfit()`
Fits a linear regression trend line to transaction amount growth over sequential index steps. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `np.polyfit(steps, amounts[:100], deg=1)`


In [5]:
steps = np.arange(100)
linear_fit = np.polyfit(steps, amounts[:100], deg=1)
print('Fitted Linear Trend Slope & Intercept [m, c]:', linear_fit.round(4))

Fitted Linear Trend Slope & Intercept [m, c]: [  1.0432 955.603 ]


### 🔹 Polynomial Evaluation with `np.polyval()`
Forecasts future transaction amounts using the fitted polynomial equation. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `np.polyval(linear_fit, future_steps)`


In [6]:
future_steps = np.array([101, 102, 103])
predictions = np.polyval(linear_fit, future_steps)
print('Forecasted Transaction Amounts for Next 3 Steps:', predictions.round(2))

Forecasted Transaction Amounts for Next 3 Steps: [1060.97 1062.01 1063.05]


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data questions explained simply with real examples.


### 🔍 Scenario: Q1: Quadratic Polynomial Trendline on Account Spending vs Age

**Approach:** Fit a 2nd-degree polynomial curve ($y = ax^2 + bx + c$) modeling transaction amount as a non-linear function of account age.
**Syntax:** `np.polyfit(account_ages[:500], amounts[:500], deg=2)`


In [7]:
quad_coeffs = np.polyfit(account_ages[:500], amounts[:500], deg=2)
residuals = amounts[:500] - np.polyval(quad_coeffs, account_ages[:500])
print('Quadratic Polynomial Coefficients [a, b, c]:', quad_coeffs.round(4))
print('Root Mean Squared Error (RMSE):', round(np.sqrt(np.mean(residuals**2)), 2))

Quadratic Polynomial Coefficients [a, b, c]: [ 1.8200000e-02 -2.7017000e+00  1.1083681e+03]
Root Mean Squared Error (RMSE): 577.6
